# 03z — Modelo final consolidado

Embalaje final:
- Re-entrenar CatBoost (best_params 03g) sobre TRAIN+VAL combinados
- Aplicar calibración ganadora (de 03i)
- Guardar modelo `final_*` en `data/data_models/`
- Generar predicciones finales sobre test
- Reporte ejecutivo `FINAL_MODEL_SUMMARY.md`

In [1]:
# [SETUP]
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, f1_score,
    confusion_matrix, brier_score_loss,
)
import json
import joblib
from pathlib import Path
from datetime import datetime

RANDOM_STATE = 42
TARGETS = ['churn_14d', 'churn_30d']
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase3_modeling' else Path.cwd()
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
DATA_MODELS = PROJECT_ROOT / 'data' / 'data_models'
INFORMES = PROJECT_ROOT / 'informes' / 'fase3_modeling'
TUNING_DIR = INFORMES / '03g_tuning_catboost'
CAL_DIR = INFORMES / '03i_calibration'

MASTER_PATH = DATA_QC / 'master_table_cutoff_v3_aggressive.parquet'
SPLITS_PATH = DATA_MODELS / 'splits_indices_cutoff.parquet'

# Cargar best_params y mejor método de calibración
best_params = {}
for target in TARGETS:
    with open(TUNING_DIR / f'best_params_{target}.json') as f:
        best_params[target] = json.load(f)
cal_metrics = pd.read_csv(CAL_DIR / 'calibration_metrics.csv')
best_cal_method = {}
for target in TARGETS:
    sub = cal_metrics[cal_metrics['target']==target].sort_values('brier').iloc[0]
    best_cal_method[target] = sub['method']
    print(f'{target}: best_calibration = {sub["method"]} (brier={sub["brier"]:.4f})')

churn_14d: best_calibration = uncalibrated (brier=0.1305)
churn_30d: best_calibration = isotonic (brier=0.1770)


In [2]:
# [EXEC] Cargar y preparar datos
master = pd.read_parquet(MASTER_PATH).reset_index(drop=True)
splits_df = pd.read_parquet(SPLITS_PATH).reset_index(drop=True)
if 'user_id' in splits_df.columns:
    splits_df = splits_df.set_index('user_id').reindex(master['user_id'].values).reset_index()

cat_cols = master.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'user_id']
for c in cat_cols:
    master[c] = master[c].astype(str).fillna('missing').replace('nan', 'missing')

X_full = master.drop(columns=['churn_14d', 'churn_30d', 'user_id'])
trainval_mask = splits_df['split'].values != 'test'
test_mask = splits_df['split'].values == 'test'
val_mask = splits_df['split'].values == 'val'

X_trainval = X_full[trainval_mask].reset_index(drop=True)
X_val = X_full[val_mask].reset_index(drop=True)
X_test = X_full[test_mask].reset_index(drop=True)
user_id_test = master.loc[test_mask, 'user_id'].reset_index(drop=True)
print(f'X_trainval={X_trainval.shape} | X_val={X_val.shape} | X_test={X_test.shape}')

X_trainval=(21420, 77) | X_val=(3781, 77) | X_test=(3780, 77)


In [3]:
# [EXEC] Re-entrenar sobre TRAIN+VAL y aplicar calibración. Predicciones finales sobre TEST.
final_metrics = []
predictions = pd.DataFrame({'user_id': user_id_test.values})

for target in TARGETS:
    print(f'\n=== Final {target} ===')
    y = master[target].astype(int)
    y_trainval, y_val, y_test = y[trainval_mask], y[val_mask], y[test_mask]

    # Re-entrenar sobre TRAIN+VAL completo. Para early stopping necesitamos eval_set,
    # pero como ya conocemos best_iteration de tuning, usamos las iterations finales.
    # Estrategia: fit sobre trainval con iterations = best_params['iterations'] sin early stopping.
    params = {**best_params[target], 'random_state': RANDOM_STATE,
              'eval_metric': 'AUC', 'verbose': False, 'thread_count': -1}
    train_pool = Pool(X_trainval, y_trainval, cat_features=cat_cols)
    test_pool = Pool(X_test, y_test, cat_features=cat_cols)
    val_pool = Pool(X_val, y_val, cat_features=cat_cols)
    model = CatBoostClassifier(**params)
    model.fit(train_pool, verbose=False)

    # Probabilidades raw
    proba_test_raw = model.predict_proba(test_pool)[:, 1]
    proba_val_raw = model.predict_proba(val_pool)[:, 1]

    # Aplicar calibración ganadora (entrenada sobre VAL)
    method = best_cal_method[target]
    if method == 'platt':
        cal = LogisticRegression(max_iter=1000)
        cal.fit(proba_val_raw.reshape(-1, 1), y_val)
        proba_test_cal = cal.predict_proba(proba_test_raw.reshape(-1, 1))[:, 1]
        joblib.dump(cal, str(DATA_MODELS / f'final_calibrator_{target}.joblib'))
    elif method == 'isotonic':
        cal = IsotonicRegression(out_of_bounds='clip')
        cal.fit(proba_val_raw, y_val)
        proba_test_cal = cal.predict(proba_test_raw)
        joblib.dump(cal, str(DATA_MODELS / f'final_calibrator_{target}.joblib'))
    else:
        proba_test_cal = proba_test_raw

    # Métricas finales
    pred = (proba_test_cal >= 0.5).astype(int)
    auc_test = float(roc_auc_score(y_test, proba_test_cal))
    auc_pr = float(average_precision_score(y_test, proba_test_cal))
    bs = float(brier_score_loss(y_test, proba_test_cal))
    ll = float(log_loss(y_test, np.clip(proba_test_cal, 1e-9, 1-1e-9)))
    f1 = float(f1_score(y_test, pred))
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    final_metrics.append({
        'target': target,
        'calibration': method,
        'auc_roc': auc_test,
        'auc_pr': auc_pr,
        'brier': bs,
        'log_loss': ll,
        'f1': f1,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'precision': float(tp/max(1, tp+fp)),
        'recall': float(tp/max(1, tp+fn)),
    })
    predictions[f'{target}_proba'] = proba_test_cal
    predictions[f'{target}_pred'] = pred
    predictions[f'{target}_true'] = y_test.values

    # Guardar modelo final
    model.save_model(str(DATA_MODELS / f'final_model_catboost_{target}.cbm'))
    print(f'  AUC test (calibrado {method}): {auc_test:.4f} | Brier: {bs:.4f} | F1: {f1:.4f}')

predictions.to_parquet(DATA_MODELS / 'final_predictions_test.parquet', index=False)
fm_df = pd.DataFrame(final_metrics)
fm_df.to_csv(INFORMES / 'final_metrics.csv', index=False)
print('\n=== Final metrics ===')
print(fm_df.round(4).to_string(index=False))


=== Final churn_14d ===


  AUC test (calibrado uncalibrated): 0.8465 | Brier: 0.1303 | F1: 0.7024

=== Final churn_30d ===


  AUC test (calibrado isotonic): 0.7948 | Brier: 0.1851 | F1: 0.6907

=== Final metrics ===
   target  calibration  auc_roc  auc_pr  brier  log_loss     f1   tn  fp  fn   tp  precision  recall
churn_14d uncalibrated   0.8465  0.8099 0.1303    0.4160 0.7024 2365 144 505  766     0.8418  0.6027
churn_30d     isotonic   0.7948  0.8076 0.1851    0.6811 0.6907 1657 330 673 1120     0.7724  0.6247


In [4]:
# [REPORT] FINAL_MODEL_SUMMARY.md
lines = [
    '# Modelo final — CatBoost tuneado y calibrado',
    '',
    f'**Fecha:** {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'**Algoritmo:** CatBoost (ganador de la comparación 03f)',
    f'**Master:** master_table_cutoff_v3_aggressive.parquet ({master.shape[1]} cols)',
    f'**Entrenamiento:** TRAIN + VAL combinados ({trainval_mask.sum():,} usuarios)',
    f'**Evaluación:** TEST ({test_mask.sum():,} usuarios, no usado en tuning ni calibración)',
    '',
    '## Métricas finales sobre TEST',
    '',
    '| target | calibration | AUC-ROC | AUC-PR | Brier | log_loss | F1 | precision | recall |',
    '|---|---|---:|---:|---:|---:|---:|---:|---:|',
]
for row in final_metrics:
    lines.append(
        f'| {row["target"]} | {row["calibration"]} | {row["auc_roc"]:.4f} | {row["auc_pr"]:.4f} | '
        f'{row["brier"]:.4f} | {row["log_loss"]:.4f} | {row["f1"]:.4f} | '
        f'{row["precision"]:.4f} | {row["recall"]:.4f} |'
    )

lines += [
    '',
    '## Confusion matrix por target',
    '',
]
for row in final_metrics:
    lines += [
        f'### {row["target"]}',
        '',
        '|  | pred=0 | pred=1 |',
        '|---|---:|---:|',
        f'| **true=0** | TN={row["tn"]} | FP={row["fp"]} |',
        f'| **true=1** | FN={row["fn"]} | TP={row["tp"]} |',
        '',
    ]

lines += [
    '## Comparación con baseline inicial (03a LightGBM defaults)',
    '',
    '| target | LightGBM baseline | CatBoost tuned + calibrado | Δ |',
    '|---|---:|---:|---:|',
    f'| churn_14d | 0.8410 (mediano v1-v3) | {final_metrics[0]["auc_roc"]:.4f} | {final_metrics[0]["auc_roc"]-0.8410:+.4f} |',
    f'| churn_30d | 0.7869 (mediano v1-v3) | {final_metrics[1]["auc_roc"]:.4f} | {final_metrics[1]["auc_roc"]-0.7869:+.4f} |',
    '',
    '## Outputs',
    '',
    '- `data/data_models/final_model_catboost_<target>.cbm` × 2 — modelos finales',
    '- `data/data_models/final_calibrator_<target>.joblib` × 2 — calibradores entrenados',
    '- `data/data_models/final_predictions_test.parquet` — predicciones test set',
    '- `informes/fase3_modeling/final_metrics.csv` — métricas finales',
    '',
    '## Próximos pasos',
    '',
    '1. Documentar resultados en memoria (`MEMORIA_DRAFT_seccion_modelado.md` §4.10).',
    '2. Cierre del bloque churn. Próxima conversación: modelo de gustos.',
]

with open(INFORMES / 'FINAL_MODEL_SUMMARY.md', 'w') as f:
    f.write('\n'.join(lines))
print('FINAL_MODEL_SUMMARY.md guardado')

FINAL_MODEL_SUMMARY.md guardado
